In [3]:
from pyspark.sql.session import SparkSession

spark = SparkSession.builder.master("local").appName('writing_data').getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/14 18:47:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/14 18:47:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/14 18:47:34 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [12]:
# it can be a single parquet file or multiple files

# reading all the parquet files in the ParquetData/ directory
# We have 2 parquet files here each have 1000 records. Total record count in df should be 2000
df = spark.read.parquet("/Users/shishir/Pyspark/SampleData/ParquetData/")

print('Total records count: ', df.count())

df.printSchema()

df.show()

Total records count:  2000
root
 |-- registration_dttm: timestamp (nullable = true)
 |-- id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- cc: string (nullable = true)
 |-- country: string (nullable = true)
 |-- birthdate: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- title: string (nullable = true)
 |-- comments: string (nullable = true)

+-------------------+---+----------+---------+--------------------+------+---------------+-------------------+--------------------+----------+---------+--------------------+--------------------+
|  registration_dttm| id|first_name|last_name|               email|gender|     ip_address|                 cc|             country| birthdate|   salary|               title|            comments|
+-------------------+---+----------+---------+---------------

In [13]:
# Using wildcard char to read all parquet files in ParquetData/ directory 
df = spark.read.parquet("/Users/shishir/Pyspark/SampleData/ParquetData/*.parquet")
df.count()     

2000

In [19]:
df = spark.createDataFrame(data = [(1, "Maheer", "male", 2000),
                                    (2, "Wafa", "male", 3000)], 
                          schema = "id int, name string, gender string, salary int")

# save the DataFrame into a Parquet file 
df.write.mode("overwrite").format("parquet").save("/Users/shishir/Pyspark/SampleData/ParquetData/dummyParquetData/")

# another way to write parquet files
# Interesing fact here is: you will generally not get a single parquet file. Instead you will see multiple files. Data is split across different files
# all the files will be created inside dummyParquetData directory. 
# If you want a single parquet file to contains all the data from DataFrame -- repartition the df to 1 partition and then write the data.
df.write.parquet("/Users/shishir/Pyspark/SampleData/ParquetData/dummyParquetData/", mode="overwrite")

In [20]:
# reading back data
df = spark.read.parquet("/Users/shishir/Pyspark/SampleData/ParquetData/dummyParquetData/")
df.show()
df.printSchema()

+---+------+------+------+
| id|  name|gender|salary|
+---+------+------+------+
|  1|Maheer|  male|  2000|
|  2|  Wafa|  male|  3000|
+---+------+------+------+

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)



Help on function display in module IPython.core.display_functions:

display(*objs, include=None, exclude=None, metadata=None, transient=None, display_id=None, raw=False, clear=False, **kwargs)
    Display a Python object in all frontends.
    
    By default all representations will be computed and sent to the frontends.
    Frontends can decide which representation is used and how.
    
    In terminal IPython this will be similar to using :func:`print`, for use in richer
    frontends see Jupyter notebook examples with rich display logic.
    
    Parameters
    ----------
    *objs : object
        The Python objects to display.
    raw : bool, optional
        Are the objects to be displayed already mimetype-keyed dicts of raw display data,
        or Python objects that need to be formatted before display? [default: False]
    include : list, tuple or set, optional
        A list of format type strings (MIME types) to include in the
        format data dict. If this is set *only* 